# Warm-up – Threads vs. Processes (and why pandas is slow)

Before we compare big-data engines, we need one fact about Python:
**the Global Interpreter Lock (GIL) lets only one thread execute Python code at a time.**

We test three kinds of work:
1. **Waiting** (chores) – nothing to compute, just waiting
2. **CPU-bound** – pure computation
3. **I/O-bound** – downloading web pages

⚠️ This notebook uses `multiprocessing` with functions defined in the notebook. That works in our
Linux dev container, but often fails on local Windows/macOS Jupyter – one more reason we use the container.

In [ ]:
"""Imports for the warm-up.

time               - perf_counter() for precise timing, sleep() to simulate waiting
os                 - cpu_count(): how many cores (= max useful processes)
threading          - Python's low-level thread module (threads share one GIL)
multiprocessing    - low-level process module (each process has its own GIL)
ThreadPoolExecutor  - high-level pool that runs tasks in threads
ProcessPoolExecutor - same interface, but runs tasks in separate processes
matplotlib.pyplot  - draws the timeline chart in section 1
"""
import time, os, threading, multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import matplotlib.pyplot as plt
print("CPU cores:", os.cpu_count())

: 

## 1. Household chores – single thread vs. threads vs. processes

Three chores take 8 s, 2 s and 4 s. Each chore only **waits** (`time.sleep`), it computes nothing.
We run them in three ways and draw a timeline of when each chore starts and ends:

- **single thread** – one chore after the other → 8 + 2 + 4 = **14 s**
- **threads** – all three at once inside one process → as long as the longest chore, **≈ 8 s**
- **processes** – all three at once in separate processes → also **≈ 8 s** (plus a little start-up time)

A sleeping thread releases the GIL, so here the GIL doesn't slow the threads down.

In [ ]:
# chore name -> how many seconds it takes
CHORES = {"walk the dog": 8, "take out the trash": 2, "get the mail": 4}

def chore(args):
    """Do one chore and report when it started and ended (relative to t0)."""
    name, seconds, t0 = args                 # one tuple, because ex.map() passes a single argument
    start = time.perf_counter() - t0         # start time, in seconds since the run began
    time.sleep(seconds)                      # waiting, not computing
    return name, start, time.perf_counter() - t0   # measured INSIDE the task

def run(mode):
    """Run all chores in the given mode and return a list of (name, start, end)."""
    t0 = time.perf_counter()                 # common start time, shared by all chores
    jobs = [(n, s, t0) for n, s in CHORES.items()]   # e.g. ("walk the dog", 8, t0)
    if mode == "single thread":
        return [chore(j) for j in jobs]      # plain loop: the next chore starts when the last one ends
    # Both pool classes have the same interface, so we just pick one
    Pool = ThreadPoolExecutor if mode == "threads" else ProcessPoolExecutor
    with Pool(len(jobs)) as ex:              # one worker per chore -> all run at the same time
        return list(ex.map(chore, jobs))     # map() hands each job to a worker; list() waits for all results

# One chart row per mode, all sharing the same time axis
fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
for ax, mode in zip(axes, ["single thread", "threads", "processes"]):
    res = run(mode)                          # [(name, start, end), ...]
    # Horizontal bars: y = chore name, length = end - start, bar begins at start
    ax.barh([r[0] for r in res], [r[2] - r[1] for r in res], left=[r[1] for r in res])
    # Total time = when the last chore finished
    ax.set_title(f"{mode}: {max(r[2] for r in res):.1f} s total"); ax.grid(axis="x")
axes[-1].set_xlabel("seconds"); plt.tight_layout(); plt.show()

**Question:** Threads and processes are equally fast here. Why doesn't the GIL matter for waiting?

The GIL only prevents two threads from executing Python code at the same time. During time.sleep() (just like during I/O, e.g. waiting for the network) a thread executes no Python code and releases the GIL. So the other threads can start their chores, and the waiting times overlap. The total time is therefore the longest chore (≈ 8 s), not the sum (14 s). Processes, with their own GIL each, can't do better, because there was no GIL bottleneck to begin with.

## 2. CPU-bound work – now the GIL hurts

In [ ]:
def cpu_heavy(n):
    """Pure computation: add up 0 + 1 + ... + (n-1) in a Python loop. No waiting at all."""
    total = 0
    for i in range(n):                       # 10 million loop steps, each one is Python code
        total += i
    return total

N, JOBS = 10**7, 4          # N = loop size per job, JOBS = how many jobs. Increase N if your machine is fast

# 1) Serial: run the 4 jobs one after another in the main thread
t = time.perf_counter(); [cpu_heavy(N) for _ in range(JOBS)]
t_serial = time.perf_counter() - t           # reference time for the speed-ups below

# 2) Threads: 4 worker threads inside this one Python process
with ThreadPoolExecutor(JOBS) as ex:
    t = time.perf_counter(); list(ex.map(cpu_heavy, [N] * JOBS))   # [N] * JOBS = [N, N, N, N] -> one job per worker
t_threads = time.perf_counter() - t          # clock stops after the "with" block, i.e. once the pool has shut down

# 3) Processes: 4 separate Python processes, each with its own interpreter
with ProcessPoolExecutor(JOBS) as ex:
    t = time.perf_counter(); list(ex.map(cpu_heavy, [N] * JOBS))   # same jobs, same interface as above
t_procs = time.perf_counter() - t

# Speed-up = serial time / parallel time (2.0x means twice as fast)
print(f"serial    {t_serial:5.2f} s")
print(f"threads   {t_threads:5.2f} s   (speed-up {t_serial/t_threads:.1f}x)")
# Upper limit: you can't run more jobs at once than you have jobs or CPU cores
print(f"processes {t_procs:5.2f} s   (speed-up {t_serial/t_procs:.1f}x, max possible ≈ {min(JOBS, os.cpu_count())}x)")

**Question:** Why do threads bring no speed-up here? What limits the process speed-up?

cpu_heavy is pure Python code, so a thread needs the GIL for every step and never releases it. The threads only take turns (interleaved), so they are no faster than serial, and slightly slower due to switching. Processes each have their own GIL and really run in parallel. The speed-up is limited by min(jobs, CPU cores) and reduced further by overhead: starting the processes and copying (pickling) data between them, since they share no memory.

## 3. I/O-bound work – threads shine

In [ ]:
import urllib.request
PAGES = ["Big_data", "Apache_Spark", "Apache_Kafka", "Apache_Hadoop", "MapReduce", "Apache_Flink",
         "Data_lake", "Data_warehouse", "Apache_Parquet", "DuckDB", "Polars_(software)", "Kufstein",
         "Innsbruck", "Tyrol_(federal_state)", "Global_interpreter_lock", "Moore's_law"]
URLS = [f"https://en.wikipedia.org/wiki/{p}" for p in PAGES]

def load(url):
    req = urllib.request.Request(url, headers={"User-Agent": "FH-Kufstein-BigData-Course/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            return len(r.read())
    except Exception as e:
        print("failed:", url, e); return 0

t = time.perf_counter(); [load(u) for u in URLS]
print(f"serial        {time.perf_counter()-t:5.2f} s")
for n in [4, 8, 16]:
    with ThreadPoolExecutor(n) as ex:
        t = time.perf_counter(); list(ex.map(load, URLS))
    print(f"{n:2d} threads    {time.perf_counter()-t:5.2f} s")

## Takeaways
| Work type | Use | Why |
|---|---|---|
| waiting / I/O | threads | the GIL is released while waiting |
| CPU-bound Python code | processes | each process has its own GIL – but no shared memory |

**Bridge to the main lab:** pandas runs most operations on one core. Polars (Rust) and DuckDB (C++)
run their work *outside* the GIL on all cores – no multiprocessing tricks needed.
Spark goes one step further: many processes on **many machines**.

*Outlook:* Python 3.13+ offers an experimental free-threaded build without the GIL (PEP 703).